In [ ]:
# Instalar clientes en caso de no tenerlos en el entorno jupyter
# !pip install pandas influxdb-client

import pandas as pd
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

In [ ]:
# --- CONFIGURACIÓN DE INFLUXDB (v2) ---
INFLUX_URL = "http://192.168.101.241:8086"       # URL o IP de tu servidor InfluxDB
INFLUX_TOKEN = "smartek-token-12345-final"      # Token de API generado en InfluxDB
INFLUX_ORG = "smartek"             # Nombre de tu Organización
INFLUX_BUCKET = "dobot"          # Bucket donde guardarás la telemetría del Dobot

# --- PARÁMETROS DEL DATASET ---
CSV_FILE = "../datos/dataset_dobot_cr5a.csv"
DEVICE_NAME = "Dobot_CR5A"                 # Se asignará como Tag

In [ ]:
print(f"📂 Cargando dataset desde '{CSV_FILE}'...")

try:
    df = pd.read_csv(CSV_FILE)
    
    # Parsear la estampa de tiempo a objetos datetime nativos
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    print(f"✅ Registros cargados: {len(df)} filas.")
    print("\nVista previa de los datos:")
    display(df.head(3))

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo '{CSV_FILE}'. Revisa la ruta.")

In [ ]:
# Inicializar cliente de InfluxDB y la API de escritura síncrona
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

puntos_batch = []
tamanio_lote = 500  # Envío agrupado de 500 filas por llamada para maximizar velocidad
total_filas = len(df)

print(f"🚀 Subiendo {total_filas} registros al bucket '{INFLUX_BUCKET}'...")

for index, row in df.iterrows():
    punto = (
        Point("dobot_telemetry")  # Measurement
        
        # --- TAGS (Metadatos de filtrado rápido) ---
        .tag("device_name", DEVICE_NAME)
        .tag("model", "CR5A")
        
        # --- FIELDS: Posición Cartesiana de la Herramienta (mm / deg) ---
        .field("pos_x", float(row["pos_x"]))
        .field("pos_y", float(row["pos_y"]))
        .field("pos_z", float(row["pos_z"]))
        .field("rot_rx", float(row["rot_rx"]))
        .field("rot_ry", float(row["rot_ry"]))
        .field("rot_rz", float(row["rot_rz"]))
        
        # --- FIELDS: Temperaturas de los 6 Servomotores (°C) ---
        .field("j1_temp", float(row["j1_temp"]))
        .field("j2_temp", float(row["j2_temp"]))
        .field("j3_temp", float(row["j3_temp"]))
        .field("j4_temp", float(row["j4_temp"]))
        .field("j5_temp", float(row["j5_temp"]))
        .field("j6_temp", float(row["j6_temp"]))
        
        # --- FIELDS: Corrientes de Consumo de los 6 Ejes (A) ---
        .field("j1_current", float(row["j1_current"]))
        .field("j2_current", float(row["j2_current"]))
        .field("j3_current", float(row["j3_current"]))
        .field("j4_current", float(row["j4_current"]))
        .field("j5_current", float(row["j5_current"]))
        .field("j6_current", float(row["j6_current"]))
        
        # --- FIELDS: Estado Operativo del Robot ---
        .field("robot_mode", int(row["robot_mode"]))
        
        # --- TIMESTAMP ---
        .time(row["timestamp"], WritePrecision.NS)
    )
    puntos_batch.append(punto)

    # Enviar al servidor al completar un lote
    if len(puntos_batch) >= tamanio_lote:
        write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=puntos_batch)
        puntos_batch = []
        print(f"   ⏳ Registros cargados: {index + 1}/{total_filas}...")

# Enviar los datos sobrantes del último lote
if puntos_batch:
    write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=puntos_batch)

print(f"\n✅ ¡Migración completada! {total_filas} filas insertadas en InfluxDB.")
client.close()